# Experiment Tracking — Model A & Model B

This notebook tracks all experiments, hyperparameter tuning, and ablation studies.

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..'))

import pandas as pd
import numpy as np
import joblib
from sklearn.metrics import accuracy_score, f1_score
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from xgboost import XGBClassifier

from src.preprocessing import load_processed_data

data = load_processed_data()
X_train, y_train = data['X_train'], data['y_train']
X_val, y_val = data['X_val'], data['y_val']
X_test, y_test = data['X_test'], data['y_test']

print(f"Train: {X_train.shape}, Val: {X_val.shape}, Test: {X_test.shape}")

## Experiment 1: Hyperparameter Sweep — Logistic Regression (C values)

In [ ]:
scaler = StandardScaler()
X_tr = scaler.fit_transform(X_train)
X_v = scaler.transform(X_val)

results = []
for C in [0.01, 0.1, 0.5, 1.0, 5.0, 10.0]:
    lr = LogisticRegression(max_iter=1000, C=C, random_state=42)
    lr.fit(X_tr, y_train)
    y_pred = lr.predict(X_v)
    acc = accuracy_score(y_val, y_pred)
    f1 = f1_score(y_val, y_pred, average='macro')
    results.append({'C': C, 'accuracy': acc, 'f1': f1})

lr_sweep = pd.DataFrame(results)
print(lr_sweep.to_string(index=False))

## Experiment 2: XGBoost — Depth & Estimator Sweep

In [ ]:
results = []
for n_est in [50, 100, 200]:
    for depth in [4, 6, 8]:
        xgb = XGBClassifier(n_estimators=n_est, max_depth=depth, learning_rate=0.1,
                            random_state=42, eval_metric='logloss', n_jobs=-1)
        xgb.fit(X_train, y_train)
        y_pred = xgb.predict(X_val)
        acc = accuracy_score(y_val, y_pred)
        f1 = f1_score(y_val, y_pred, average='macro')
        results.append({'n_estimators': n_est, 'max_depth': depth, 'accuracy': acc, 'f1': f1})

xgb_sweep = pd.DataFrame(results)
print(xgb_sweep.sort_values('f1', ascending=False).to_string(index=False))

## Experiment 3: K-Means — Varying Number of Clusters

In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

X_scaled = scaler.fit_transform(X_train)
results = []
for k in [2, 3, 4, 5, 8]:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_scaled)
    sil = silhouette_score(X_scaled, km.labels_, sample_size=min(5000, len(X_scaled)))
    results.append({'k': k, 'silhouette': sil, 'inertia': km.inertia_})

km_sweep = pd.DataFrame(results)
print(km_sweep.to_string(index=False))

## Summary: Best Configurations

| Model | Best Params | Val F1 |
|-------|-------------|--------|
| Logistic Regression | See sweep above | - |
| XGBoost | See sweep above | - |
| K-Means | See silhouette above | - |

Run cells above to populate results.

# Experiment Tracking — Model A & Model B

This notebook tracks all experiments, hyperparameter tuning, and ablation studies.